In [47]:
import numpy as np

from ngio import open_ome_zarr_container, open_image, open_label, create_empty_ome_zarr
from pathlib import Path

In [4]:
source_zarr_path = Path(
    "training_data.zarr/", 
)

In [5]:
source_zarr_path.exists()

True

In [8]:
excluded_indices = {
    7, 12, 30, 33, 34, 35, 38, 41, 42, 43, 45, 46, 
    49, 50, 56, 59, 62, 63, 72, 73, 79, 82, 86, 87,
    89, 90, 92, 93, 94, 95, 96, 100, 102, 112, 113,
    116, 123, 127, 134, 138, 144, 148, 167, 187,
    192, 194, 198, 200, 202, 203, 204, 214, 215,
    216, 217, 218, 223, 229, 231, 232, 233, 234, 
    237, 247, 251, 253, 258, 261, 274, 279, 282,
    283, 286, 369, 466, 471, 521, 559, 564, 566,
    618, 692, 702, 705, 707, 724, 764, 765, 771,
    793, 794, 844, 945, 946, 953, 954, 968, 969,
    970, 971, 973, 976, 979, 980, 981, 988, 990,
    991, 995, 998, 999, 1002, 1005, 1013, 1014,
    1016, 1017, 1018, 1020, 1021, 1022, 1023, 1024,
    1025, 1026, 1027, 1028, 1031, 1032, 1033, 1035,
    1042, 1043, 1045, 1052, 1053, 1054,
    1121, 1122, 1160, 1162, 1179, 2694, 
}

In [10]:
source_zarr_group = open_ome_zarr_container(
    source_zarr_path, 
    mode="r",
)

In [12]:
source_zarr_group.get_image()

Image(path=s0, Dimensions(t: 3949, y: 1200, x: 1200))

In [14]:
source_zarr_group.get_label("mask")

Label(path=0, Dimensions(t: 3949, y: 1200, x: 1200))

In [22]:
valid_indices = [i for i in range(source_zarr_group.get_image().shape[0]) if i not in excluded_indices]

In [25]:
len(valid_indices), len(excluded_indices)

(3806, 143)

In [28]:
# randomize indices
valid_indices = np.array(valid_indices)
np.random.shuffle(valid_indices)

validation_split = 0.1
# split into train/val indices
num_validation = int(len(valid_indices) * validation_split)
train_indices = valid_indices[num_validation:]
val_indices = valid_indices[:num_validation]
len(train_indices), len(val_indices)

(3426, 380)

In [33]:
out_path_train = Path(
    "/Volumes/tachyon/groups/scratch/gmicro_prefect/ggrossha/ggrossha_SWI/training_data",
)
out_path_train.exists()

True

In [48]:

out_image_train = create_empty_ome_zarr(
    out_path_train / "accumulated_train.zarr",
    shape=(len(train_indices),) + source_zarr_group.get_image().shape[1:],
    chunks=(1,) + source_zarr_group.get_image().shape[1:],
    xy_pixelsize=1.0,
    levels=1,
    overwrite=True,
    dimension_separator=".",
    axes_names=["t", "y", "x"],
)

# out_image_train = source_zarr_group.derive_image(
#     out_path_train / "accumulated_train.zarr",
#     shape=(len(train_indices),) + source_zarr_group.get_image().shape[1:],
#     overwrite=True,
# )
out_image_train.get_image()

Image(path=0, Dimensions(t: 3426, y: 1200, x: 1200))

In [54]:
out_label_train = out_image_train.derive_label(
    name="mask",
    overwrite=True,
)
out_label_train

Label(path=0, Dimensions(t: 3426, y: 1200, x: 1200))

In [56]:
out_image_data = out_image_train.get_image()
out_image_data.zarr_array
out_label_train.zarr_array

<zarr.core.Array '/labels/mask/0' (3426, 1200, 1200) uint32>

In [62]:
source_image_data = source_zarr_group.get_image()
source_label_data = source_zarr_group.get_label("mask")


In [ ]:
for i, idx in enumerate(train_indices):
    # print(f"Writing {idx} into {i} ({source_image_data.zarr_array[idx].shape} into {out_image_data.zarr_array[i].shape})")
    out_image_data.zarr_array[i] = source_image_data.zarr_array[idx]
    out_label_train.zarr_array[i] = source_label_data.zarr_array[idx]


In [ ]:
out_image_val = create_empty_ome_zarr(
    out_path_train / "accumulated_val.zarr",
    shape=(len(val_indices),) + source_zarr_group.get_image().shape[1:],
    chunks=(1,) + source_zarr_group.get_image().shape[1:],
    xy_pixelsize=1.0,
    levels=1,
    overwrite=True,
    dimension_separator=".",
    axes_names=["t", "y", "x"],    
)

In [ ]:
out_label_val = out_image_val.derive_label(
    name="mask",
    overwrite=True,    
)

In [ ]:
out_image_val_data = out_image_val.get_image()

In [ ]:
for i, idx in enumerate(val_indices):
    # print(f"Writing {idx} into {i} ({source_image_data.zarr_array[idx].shape} into {out_image_data.zarr_array[i].shape})")
    out_image_val_data.zarr_array[i] = source_image_data.zarr_array[idx]
    out_label_val.zarr_array[i] = source_label_data.zarr_array[idx]
